# 00 · Setup, environment and the pipeline map

These notebooks take the pipeline apart, one stage per notebook, so each model / rule can be inspected, measured and
replaced on its own:

| # | notebook | stage | input | output (stage table) |
|---|----------|-------|-------|----------------------|
| 01 | `01_video_to_frames` | decode, sub-sample, grass mask / ROI | mp4 | `frames_index.csv` |
| 02 | `02_player_detection` | YOLOX tiles → boxes (people + ball candidates) | frames | `detections.csv` |
| 03 | `03_pitch_calibration` | homography image ↔ pitch (static / flow / PTZ) | frames + people boxes | `homography.csv` |
| 04 | `04_tracking_and_teams` | IDs across frames, person type, kit → team / role | detections + homography | `tracks.csv` |
| 05 | `05_ball_detection_tracking` | ball tracker (Kalman + zoom + scan) → cleaned trajectory | frames + people | `ball_raw.csv`, `ball.csv` |
| 06 | `06_integrated_perception` | the real loop (`run_perception`) on the same window, cross-checked with 02–05 | mp4 | `perception/` + `benchmarks.csv` |
| 07 | `07_analytics_events_stats` | possession → events → xG → team / player stats | tracks + ball | tables |
| 08 | `08_model_training_cycle` | export data → label → train → evaluate → deploy a detector | a full run | dataset, scores, config |

**Why stages with tables in between:** you can swap one model (e.g. a fine-tuned YOLO in 02) and re-run only the
notebooks downstream on the new table, and 06 tells you whether the integrated loop agrees with the stage-by-stage
result. 06 also appends a row of KPIs to `outputs/_pipeline_nb/benchmarks.csv`, the log of your experiments.

Edit **`notebooks/pipeline/params.yaml`** (video, match setup, time window) once; every notebook reads it.

In [ ]:
import sys, os, time, json, warnings
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "football_eval").is_dir())
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.precision", 3); pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30); pd.set_option("display.max_rows", 60)
plt.rcParams.update({"figure.dpi": 90, "axes.spines.top": False, "axes.spines.right": False})
from football_eval import nbtools as nbt

P = nbt.load_params()            # notebooks/pipeline/params.yaml (or $FVE_NB_PARAMS)
ctx = nbt.build_context(P)       # config <- match setup, restricted to the window
cfg = ctx.cfg

## 1 · Where will the models run?
`accel.describe()` is what the detector sees: the NVIDIA GPU (driver), the onnxruntime execution providers and the PyTorch device. `accel.doctor()` (fix branch) goes further when CUDA is listed but not used.

In [ ]:
from football_eval import accel
print(accel.describe())
try:
    import onnxruntime as ort
    print("\nonnxruntime", ort.__version__, "| device:", ort.get_device(), "| providers:", ort.get_available_providers())
except ImportError:
    print("onnxruntime not installed -> cv2.dnn")

## 2 · The configuration actually used
Defaults ← `config` file ← match setup (`apply_to_config`) ← the window from `params.yaml`. These are the values the stages below run with; change them here (e.g. `cfg.detection.tile = 800`) to experiment in one notebook, or in the config file to change all of them.

In [ ]:
nbt.config_table(cfg)

## 3 · Video, window and expected cost

In [ ]:
info = ctx.info
n_an = int(sum(b - a for a, b in ctx.windows) * cfg.video.analysis_fps)
print(f"source fps {info.fps:.2f}, analysis fps {cfg.video.analysis_fps:g} -> every {ctx.step}th frame")
print(f"window {ctx.windows} -> {n_an} analysed frames")
full = sum(b - a for a, b in (cfg.match.halves or [(0, info.duration_s)])) * cfg.video.analysis_fps
print(f"whole match: {int(full)} analysed frames; at 5 fps processing that is {full / 5 / 3600:.1f} h, at 20 fps {full / 20 / 3600:.1f} h")
f, t, img = nbt.probe_frame(ctx)
nbt.show(img, f"probe frame {f} (t = {t:.1f} s)");

## 4 · Match setup facts
What the human told the pipeline (periods, direction, kits, calibration, match sheet). Everything downstream depends on these.

In [ ]:
if ctx.setup:
    s = ctx.setup
    print("teams       :", [t.get("name") for t in s["teams"]])
    print("periods     :", s.get("periods"))
    print("kits        :", {k: len(v) for k, v in s["kits"].items()})
    print("calibration :", s["calibration"].get("mode"), "| landmark frames:",
          [(round(f["t"], 1), len(f["points"])) for f in s["calibration"].get("landmark_frames", [])])
    print("match sheet :", s.get("final_score"), [(e.get("type"), e.get("minute"), e.get("team")) for e in s.get("known_events", [])])
else:
    print("no match setup: run scripts/match_ui.py setup for reliable teams / calibration")